# Model Training - ChatKasir

- Nama: Achmad Rif'an
- Bagian: AI-1 (Model Architect)

## 1. Import Library

In [10]:
import os
import json
import numpy as np
import tensorflow as tf

from tensorflow.keras.layers import Input, Embedding, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model

## 2. Memuat Konfigurasi

In [ ]:
# Memuat konfigurasi arsitektur dari file JSON
config_path = "..\\assets\\data\\model_config.json"
with open(config_path, "r") as f:
    config = json.load(f)

# Mengambil variabel penting
VOCAB_SIZE = config['vocab_size']
MAX_LENGTH = config['max_length']
NUM_TAGS = config['num_product_tags']

print(f"Konfigurasi dimuat: Vocab={VOCAB_SIZE}, Max Length={MAX_LENGTH}, Num Tags={NUM_TAGS}")

Konfigurasi dimuat: Vocab=5000, Max Length=64, Num Tags=3


## 3. Data Loading

In [6]:
# Memuat dataset yang sudah dibagi (Train, Val, Test)
data_path = "..\\assets\\data\\dataset_chatkasir.npz"
data = np.load(data_path)

# Ekstrak data Training
X_train = data['X_train']
Y_prod_train = data['Y_prod_train']
Y_qty_train = data['Y_qty_train']
Y_price_train = data['Y_price_train']

# Ekstrak data Validation
X_val = data['X_val']
Y_prod_val = data['Y_prod_val']
Y_qty_val = data['Y_qty_val']
Y_price_val = data['Y_price_val']

print(f"Dataset dimuat: Training={len(X_train)} baris, Validation={len(X_val)} baris")

Dataset dimuat: Training=80400 baris, Validation=10050 baris


In [ ]:
# Mengonversi ke tf.data.Dataset untuk efisiensi training
BATCH_SIZE = 32 # Jumlah data yang diproses sekali epoch

def create_tf_dataset(X, y_prod, y_qty, y_price, is_training=True):
    # Gabungkan Input (X) dengan 3 Target (Y)
    ds = tf.data.Dataset.from_tensor_slices((X, (y_prod, y_qty, y_price)))
    
    if is_training:
        ds = ds.shuffle(10000) # Acak data agar model tidak menghafal urutan
    
    # Ambil data per batch dan siapkan batch berikutnya di latar belakang (prefetch)
    # AUTOTUNE = otomatis mengatur penggunaan CPU/GPU
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

# Buat objek dataset untuk Training dan Validation
train_ds = create_tf_dataset(X_train, Y_prod_train, Y_qty_train, Y_price_train)
val_ds = create_tf_dataset(X_val, Y_prod_val, Y_qty_val, Y_price_val, is_training=False)

print("Objek tf.data.Dataset berhasil dibuat")

Objek tf.data.Dataset berhasil dibuat


## 4. Re-build Model

In [11]:
# Custom Layer Transformer dengan dukungan Masking
class TransformerEncoder(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super(TransformerEncoder, self).__init__(**kwargs)
        self.supports_masking = True # Pastikan layer mendukung penanda padding
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim) 
        self.ffn = tf.keras.Sequential([Dense(ff_dim, activation="relu"), Dense(embed_dim)])  
        self.layernorm1 = LayerNormalization(epsilon=1e-6) 
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)  
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training=False, mask=None):
        # Gunakan mask agar attention mengabaikan token padding [PAD]
        padding_mask = tf.cast(mask[:, tf.newaxis, :], dtype=tf.int32) if mask is not None else None
        
        attn_output = self.att(inputs, inputs, attention_mask=padding_mask)  
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)  

        ffn_output = self.ffn(out1) 
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# Fungsi arsitektur model
def model_transformer(vocab_size, max_length, num_product_tags):
    embed_dim = 64  # Dimensi representasi kata
    num_heads = 4   # Jumlah mekanisme attention
    ff_dim = 128    # Kapasitas memori internal
    
    # Input Layer
    inputs = Input(shape=(max_length,), name="input_ids")  
    
    # Shared Backbone (Transformer)
    # mask_zero=True sangat penting untuk menangani padding secara otomatis
    x = Embedding(input_dim=vocab_size, output_dim=embed_dim, mask_zero=True)(inputs)  
    x = TransformerEncoder(embed_dim, num_heads, ff_dim)(x) 
    x_pooled = GlobalAveragePooling1D()(x) # Ringkasan kalimat untuk regresi
    
    # Cabang 1: Produk (NER) - Memprediksi tag untuk setiap kata
    branch_product = Dense(64, activation='relu')(x)
    output_product = Dense(num_product_tags, activation='softmax', name="product_tags")(branch_product)
    
    # Cabang 2: Jumlah (Quantity) - Regresi nilai angka jumlah pesanan
    branch_quantity = Dense(32, activation='relu')(x_pooled)
    output_quantity = Dense(1, activation='relu', name="quantity")(branch_quantity)
    
    # Cabang 3: Harga (Price) - Regresi nilai harga satuan
    branch_price = Dense(32, activation='relu')(x_pooled)
    output_price = Dense(1, activation='relu', name="price")(branch_price)
    
    return Model(inputs=inputs, outputs=[output_product, output_quantity, output_price])

# Merakit model menggunakan parameter dari konfigurasi Tahap 1
model = model_transformer(
    vocab_size=VOCAB_SIZE,
    max_length=MAX_LENGTH,
    num_product_tags=NUM_TAGS
)

# Tampilkan ringkasan arsitektur
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_ids           │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 64, 64)    │    320,000 │ input_ids[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 64)        │          0 │ input_ids[0][0]   │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encoder │ (None, 64, 64)    │     83,200 │ embedding[0][0],  │
│ (TransformerEncode… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ transformer_enco… │
│ (GlobalAveragePool… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64, 64)    │      4,160 │ transformer_enco… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 32)        │      2,080 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 32)        │      2,080 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ product_tags        │ (None, 64, 3)     │        195 │ dense_2[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ quantity (Dense)    │ (None, 1)         │         33 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ price (Dense)       │ (None, 1)         │         33 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 411,781 (1.57 MB)

 Trainable params: 411,781 (1.57 MB)

 Non-trainable params: 0 (0.00 B)

## 5. Loss Function & Optimizer

In [12]:
# Custom Loss untuk harga satuan
class MaskedPriceLoss(tf.keras.losses.Loss):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        # Menggunakan Mean Squared Error sebagai dasar perhitungan
        self.mse = tf.keras.losses.MeanSquaredError(reduction='none')

    def call(self, y_true, y_pred):
        # Buat masker: abaikan jika y_true bernilai -1
        mask = tf.cast(tf.not_equal(y_true, -1.0), tf.float32)
        
        # Hitung MSE mentah
        loss = self.mse(y_true, y_pred)
        
        # Kalikan loss dengan masker (loss jadi 0 untuk data bernilai -1)
        masked_loss = loss * mask
        
        # Kembalikan rata-rata loss hanya dari data yang valid
        return tf.reduce_sum(masked_loss) / (tf.reduce_sum(mask) + 1e-7)

# Inisialisasi Loss Function untuk tiap cabang
# Sparse karena label produk berbentuk angka ID (0, 1, 2)
loss_fn_product = tf.keras.losses.SparseCategoricalCrossentropy()
loss_fn_quantity = tf.keras.losses.MeanSquaredError()
loss_fn_price = MaskedPriceLoss()

# Inisialisasi Optimizer
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# Variabel untuk Dynamic Loss Weighting
# bobot awal seimbang (1.0) untuk ketiga tugas
w_product = tf.Variable(1.0, trainable=False, name="w_prod")
w_quantity = tf.Variable(1.0, trainable=False, name="w_qty")
w_price = tf.Variable(1.0, trainable=False, name="w_price")